# 🏋️ Member 2 - YOLOv8 Pose Training (Kaggle)

> **Mục tiêu:** Fine-tune YOLOv8-Pose trên dataset squat để detect keypoints
> 
> **Platform:** Kaggle GPU (P100/T4)
> **Phase:** 2 (Model Training)

---

## 📋 Nhiệm vụ trong notebook này

1. **Setup Environment** - Cài Ultralytics, upload dataset
2. **Sanity Check** - Test pipeline với 5 epochs
3. **Full Training** - Train YOLOv8-Pose 50-100 epochs
4. **Evaluation** - Đánh giá mAP, confusion matrix
5. **Inference Test** - Test trên sample images
6. **Export Model** - Lưu weights để deploy

---

**Target Metrics:**
- Accuracy ≥ 85%
- mAP50 ≥ 0.8
- Inference < 150ms/frame

---

**Created:** 2026-04-08  
**Phase:** 2 (Training)  
**Member:** ML Engineer (CV)

## 1️⃣ Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install Ultralytics (YOLOv8)
!pip install -q ultralytics opencv-python-headless matplotlib seaborn pandas tqdm pyyaml

In [ ]:
# Import libraries
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
import yaml
import shutil
import warnings
warnings.filterwarnings('ignore')

# Ultralytics
from ultralytics import YOLO
import torch

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## 📁 Dataset Setup

**Option 1:** Upload dataset as Kaggle Dataset
1. Nén folder `cv_dataset` thành ZIP
2. Tạo Kaggle Dataset mới, upload ZIP
3. Add vào notebook qua "Add Data"

**Option 2:** Upload trực tiếp
1. Upload `cv_dataset.zip` vào Kaggle Input
2. Unzip vào working directory

In [ ]:
# ========================================
# 📌 CẤU HÌNH ĐƯỜNG DẪN DATASET
# ========================================

# Option 1: Dataset đã upload như Kaggle Dataset
# Uncomment và sửa tên dataset của bạn
# DATASET_INPUT = Path('/kaggle/input/your-dataset-name/cv_dataset')

# Option 2: Unzip từ file ZIP
# Uncomment nếu cần unzip
# !unzip -q /kaggle/input/your-zip-file/cv_dataset.zip -d /kaggle/working/

# Working directory cho training output
WORK_DIR = Path('/kaggle/working')
DATASET_ROOT = WORK_DIR / 'cv_dataset'

# Nếu dataset ở input, copy sang working (YOLO cần write access)
# if DATASET_INPUT.exists() and not DATASET_ROOT.exists():
#     print("📋 Copying dataset to working directory...")
#     shutil.copytree(DATASET_INPUT, DATASET_ROOT)
#     print("✅ Dataset copied!")

print(f"Dataset root: {DATASET_ROOT}")
print(f"Exists: {DATASET_ROOT.exists()}")

In [ ]:
# Verify dataset structure
DATA_YAML = DATASET_ROOT / 'data.yaml'

required_dirs = [
    DATASET_ROOT / 'train' / 'images',
    DATASET_ROOT / 'train' / 'labels',
    DATASET_ROOT / 'val' / 'images',
    DATASET_ROOT / 'val' / 'labels'
]

print("📁 Checking dataset structure...")
all_ok = True

for d in required_dirs:
    status = "✅" if d.exists() else "❌"
    print(f"  {status} {d}")
    if not d.exists():
        all_ok = False

if DATA_YAML.exists():
    print(f"  ✅ {DATA_YAML}")
else:
    print(f"  ❌ {DATA_YAML}")
    all_ok = False

if all_ok:
    print("\n✅ Dataset structure verified!")
else:
    print("\n⚠️ Dataset structure incomplete. Please check paths.")

In [ ]:
# Load and display data.yaml
if DATA_YAML.exists():
    with open(DATA_YAML, 'r') as f:
        data_config = yaml.safe_load(f)
    
    print("📋 data.yaml Configuration:")
    print("-"*40)
    for key, value in data_config.items():
        print(f"{key}: {value}")
else:
    print("⚠️ data.yaml not found! Creating default config...")

In [ ]:
# Create/Update data.yaml với absolute paths cho Kaggle
# QUAN TRỌNG: YOLO cần absolute paths

data_yaml_content = f"""
# Dataset configuration for YOLOv8-Pose
# Auto-generated for Kaggle

path: {DATASET_ROOT}  # dataset root dir
train: train/images   # train images (relative to path)
val: val/images       # val images (relative to path)
test: test/images     # test images (optional)

# Keypoints
kpt_shape: [18, 3]  # [num_keypoints, dims (x,y,visibility)]

# Skeleton connections for visualization
flip_idx: [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15, 17]

# Classes
names:
  0: person
"""

# Lưu data.yaml
with open(DATA_YAML, 'w') as f:
    f.write(data_yaml_content.strip())

print("✅ data.yaml created/updated!")
print("\n" + data_yaml_content)

## 2️⃣ Sanity Check - Quick Training Test

In [ ]:
# Load pretrained YOLOv8 Pose model
# Options: yolov8n-pose, yolov8s-pose, yolov8m-pose, yolov8l-pose, yolov8x-pose

MODEL_NAME = 'yolov8n-pose'  # nano - fastest, good for testing
# MODEL_NAME = 'yolov8s-pose'  # small - better accuracy
# MODEL_NAME = 'yolov8m-pose'  # medium - balanced

model = YOLO(f'{MODEL_NAME}.pt')
print(f"✅ Loaded pretrained model: {MODEL_NAME}")

In [ ]:
# Sanity check: Train for 5 epochs to verify pipeline
print("🔄 Running sanity check (5 epochs)...")
print("This may take a few minutes...\n")

results_sanity = model.train(
    data=str(DATA_YAML),
    epochs=5,
    imgsz=640,
    batch=16,
    device=0,  # GPU
    workers=4,
    project=str(WORK_DIR / 'runs' / 'sanity_check'),
    name='yolov8_pose_sanity',
    exist_ok=True,
    verbose=True
)

print("\n✅ Sanity check complete!")

In [ ]:
# Check sanity results
print("📊 Sanity Check Results:")
print("-"*40)

if hasattr(results_sanity, 'results_dict'):
    for key, value in results_sanity.results_dict.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

## 3️⃣ Full Training

In [ ]:
# ========================================
# 📌 TRAINING CONFIGURATION
# ========================================

TRAINING_CONFIG = {
    # Model
    'model': 'yolov8n-pose.pt',  # hoặc 'yolov8s-pose.pt' cho accuracy cao hơn
    
    # Training params
    'epochs': 100,          # Số epochs (50-100 recommended)
    'imgsz': 640,           # Image size
    'batch': 16,            # Batch size (giảm nếu OOM)
    'patience': 20,         # Early stopping patience
    
    # Optimizer
    'optimizer': 'AdamW',
    'lr0': 0.01,            # Initial learning rate
    'lrf': 0.01,            # Final learning rate (lr0 * lrf)
    'momentum': 0.937,
    'weight_decay': 0.0005,
    
    # Augmentation
    'hsv_h': 0.015,         # Hue augmentation
    'hsv_s': 0.7,           # Saturation augmentation  
    'hsv_v': 0.4,           # Value augmentation
    'degrees': 10.0,        # Rotation
    'translate': 0.1,       # Translation
    'scale': 0.5,           # Scale
    'shear': 0.0,           # Shear
    'flipud': 0.0,          # Flip up-down (not for pose)
    'fliplr': 0.5,          # Flip left-right
    'mosaic': 1.0,          # Mosaic augmentation
    'mixup': 0.0,           # MixUp augmentation
    
    # Device
    'device': 0,            # GPU
    'workers': 4,
    
    # Saving
    'save': True,
    'save_period': 10,      # Save checkpoint every N epochs
}

print("📋 Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Load fresh model for full training
model_full = YOLO(TRAINING_CONFIG['model'])

print(f"\n🚀 Starting full training...")
print(f"Model: {TRAINING_CONFIG['model']}")
print(f"Epochs: {TRAINING_CONFIG['epochs']}")
print(f"Batch size: {TRAINING_CONFIG['batch']}")
print("\nThis will take a while. Go grab some coffee! ☕\n")

In [ ]:
# Full training
results = model_full.train(
    data=str(DATA_YAML),
    epochs=TRAINING_CONFIG['epochs'],
    imgsz=TRAINING_CONFIG['imgsz'],
    batch=TRAINING_CONFIG['batch'],
    patience=TRAINING_CONFIG['patience'],
    
    optimizer=TRAINING_CONFIG['optimizer'],
    lr0=TRAINING_CONFIG['lr0'],
    lrf=TRAINING_CONFIG['lrf'],
    momentum=TRAINING_CONFIG['momentum'],
    weight_decay=TRAINING_CONFIG['weight_decay'],
    
    hsv_h=TRAINING_CONFIG['hsv_h'],
    hsv_s=TRAINING_CONFIG['hsv_s'],
    hsv_v=TRAINING_CONFIG['hsv_v'],
    degrees=TRAINING_CONFIG['degrees'],
    translate=TRAINING_CONFIG['translate'],
    scale=TRAINING_CONFIG['scale'],
    shear=TRAINING_CONFIG['shear'],
    flipud=TRAINING_CONFIG['flipud'],
    fliplr=TRAINING_CONFIG['fliplr'],
    mosaic=TRAINING_CONFIG['mosaic'],
    mixup=TRAINING_CONFIG['mixup'],
    
    device=TRAINING_CONFIG['device'],
    workers=TRAINING_CONFIG['workers'],
    
    save=TRAINING_CONFIG['save'],
    save_period=TRAINING_CONFIG['save_period'],
    
    project=str(WORK_DIR / 'runs' / 'pose'),
    name='yolov8_pose_full',
    exist_ok=True,
    verbose=True,
    plots=True
)

print("\n" + "="*50)
print("✅ TRAINING COMPLETE!")
print("="*50)

## 4️⃣ Training Results & Evaluation

In [ ]:
# Find best model weights
RUNS_DIR = WORK_DIR / 'runs' / 'pose' / 'yolov8_pose_full'
BEST_WEIGHTS = RUNS_DIR / 'weights' / 'best.pt'
LAST_WEIGHTS = RUNS_DIR / 'weights' / 'last.pt'

print("📁 Training outputs:")
if BEST_WEIGHTS.exists():
    print(f"  ✅ Best weights: {BEST_WEIGHTS}")
    print(f"     Size: {BEST_WEIGHTS.stat().st_size / 1024 / 1024:.2f} MB")
if LAST_WEIGHTS.exists():
    print(f"  ✅ Last weights: {LAST_WEIGHTS}")

In [ ]:
# Display training curves
results_csv = RUNS_DIR / 'results.csv'

if results_csv.exists():
    df_results = pd.read_csv(results_csv)
    df_results.columns = df_results.columns.str.strip()  # Remove whitespace
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Box Loss
    if 'train/box_loss' in df_results.columns:
        axes[0, 0].plot(df_results['epoch'], df_results['train/box_loss'], label='Train')
        if 'val/box_loss' in df_results.columns:
            axes[0, 0].plot(df_results['epoch'], df_results['val/box_loss'], label='Val')
        axes[0, 0].set_title('Box Loss')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].legend()
    
    # Pose Loss
    if 'train/pose_loss' in df_results.columns:
        axes[0, 1].plot(df_results['epoch'], df_results['train/pose_loss'], label='Train')
        if 'val/pose_loss' in df_results.columns:
            axes[0, 1].plot(df_results['epoch'], df_results['val/pose_loss'], label='Val')
        axes[0, 1].set_title('Pose Loss')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].legend()
    
    # Keypoint Loss
    if 'train/kobj_loss' in df_results.columns:
        axes[0, 2].plot(df_results['epoch'], df_results['train/kobj_loss'], label='Train')
        if 'val/kobj_loss' in df_results.columns:
            axes[0, 2].plot(df_results['epoch'], df_results['val/kobj_loss'], label='Val')
        axes[0, 2].set_title('Keypoint Obj Loss')
        axes[0, 2].set_xlabel('Epoch')
        axes[0, 2].legend()
    
    # mAP50
    if 'metrics/mAP50(B)' in df_results.columns:
        axes[1, 0].plot(df_results['epoch'], df_results['metrics/mAP50(B)'], 'g-')
        axes[1, 0].set_title('mAP50 (Box)')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylim(0, 1)
        axes[1, 0].axhline(y=0.8, color='r', linestyle='--', label='Target: 0.8')
        axes[1, 0].legend()
    
    # mAP50-95
    if 'metrics/mAP50-95(B)' in df_results.columns:
        axes[1, 1].plot(df_results['epoch'], df_results['metrics/mAP50-95(B)'], 'b-')
        axes[1, 1].set_title('mAP50-95 (Box)')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylim(0, 1)
    
    # Precision & Recall
    if 'metrics/precision(B)' in df_results.columns:
        axes[1, 2].plot(df_results['epoch'], df_results['metrics/precision(B)'], label='Precision')
        if 'metrics/recall(B)' in df_results.columns:
            axes[1, 2].plot(df_results['epoch'], df_results['metrics/recall(B)'], label='Recall')
        axes[1, 2].set_title('Precision & Recall')
        axes[1, 2].set_xlabel('Epoch')
        axes[1, 2].set_ylim(0, 1)
        axes[1, 2].axhline(y=0.85, color='r', linestyle='--', label='Target: 0.85')
        axes[1, 2].legend()
    
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ results.csv not found")

In [ ]:
# Final metrics summary
if results_csv.exists():
    df_results = pd.read_csv(results_csv)
    df_results.columns = df_results.columns.str.strip()
    
    print("="*50)
    print("📊 FINAL TRAINING METRICS")
    print("="*50)
    
    last_row = df_results.iloc[-1]
    
    metrics_to_show = [
        ('metrics/precision(B)', 'Precision'),
        ('metrics/recall(B)', 'Recall'),
        ('metrics/mAP50(B)', 'mAP50'),
        ('metrics/mAP50-95(B)', 'mAP50-95'),
        ('metrics/mAP50(P)', 'Pose mAP50'),
        ('metrics/mAP50-95(P)', 'Pose mAP50-95'),
    ]
    
    print(f"\nEpoch: {int(last_row.get('epoch', 0))}")
    print("-"*30)
    
    for col, name in metrics_to_show:
        if col in last_row:
            value = last_row[col]
            status = "✅" if value >= 0.85 else "⚠️" if value >= 0.7 else "❌"
            print(f"{status} {name}: {value:.4f}")
    
    print("\n" + "="*50)

## 5️⃣ Validation & Inference Test

In [ ]:
# Load best model
if BEST_WEIGHTS.exists():
    best_model = YOLO(str(BEST_WEIGHTS))
    print(f"✅ Loaded best model: {BEST_WEIGHTS}")
else:
    print("⚠️ Best weights not found, using last weights")
    best_model = YOLO(str(LAST_WEIGHTS))

In [ ]:
# Run validation on test set (if available) or val set
print("🔄 Running validation...")

val_results = best_model.val(
    data=str(DATA_YAML),
    split='val',  # or 'test' if available
    batch=16,
    imgsz=640,
    verbose=True
)

print("\n✅ Validation complete!")

In [ ]:
# Inference on sample images
def visualize_predictions(model, images_dir, n_samples=6):
    """Run inference and visualize predictions."""
    
    image_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))
    if not image_files:
        print("⚠️ No images found!")
        return
    
    n_samples = min(n_samples, len(image_files))
    sample_images = np.random.choice(image_files, n_samples, replace=False)
    
    cols = 3
    rows = (n_samples + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(sample_images):
        # Run inference
        results = model.predict(str(img_path), verbose=False)
        
        # Get annotated image
        annotated = results[0].plot()
        annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        
        # Display
        axes[idx].imshow(annotated)
        axes[idx].set_title(f"{img_path.name}", fontsize=10)
        axes[idx].axis('off')
    
    for idx in range(n_samples, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'prediction_samples.png', dpi=150, bbox_inches='tight')
    plt.show()

# Visualize predictions on validation images
print("🖼️ Visualizing predictions on validation samples...")
VAL_IMAGES = DATASET_ROOT / 'val' / 'images'
if VAL_IMAGES.exists():
    visualize_predictions(best_model, VAL_IMAGES, n_samples=6)

## 6️⃣ Inference Speed Benchmark

In [ ]:
import time

def benchmark_inference(model, images_dir, n_runs=50):
    """Benchmark inference speed."""
    
    image_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))
    if not image_files:
        print("⚠️ No images found!")
        return
    
    # Warmup
    print("Warming up...")
    for _ in range(5):
        model.predict(str(image_files[0]), verbose=False)
    
    # Benchmark
    times = []
    sample_images = np.random.choice(image_files, min(n_runs, len(image_files)), replace=True)
    
    print(f"Running {len(sample_images)} inference passes...")
    for img_path in tqdm(sample_images):
        start = time.time()
        model.predict(str(img_path), verbose=False)
        end = time.time()
        times.append((end - start) * 1000)  # Convert to ms
    
    times = np.array(times)
    
    print("\n" + "="*50)
    print("⏱️ INFERENCE SPEED BENCHMARK")
    print("="*50)
    print(f"Mean latency: {times.mean():.2f} ms")
    print(f"Std:          {times.std():.2f} ms")
    print(f"Min:          {times.min():.2f} ms")
    print(f"Max:          {times.max():.2f} ms")
    print(f"Median:       {np.median(times):.2f} ms")
    print(f"FPS:          {1000/times.mean():.1f}")
    
    target = 150  # ms
    if times.mean() < target:
        print(f"\n✅ Target met! ({times.mean():.1f}ms < {target}ms)")
    else:
        print(f"\n⚠️ Target NOT met ({times.mean():.1f}ms > {target}ms)")
    print("="*50)
    
    return times

# Run benchmark
if VAL_IMAGES.exists():
    inference_times = benchmark_inference(best_model, VAL_IMAGES, n_runs=50)

In [ ]:
# Plot inference time distribution
if 'inference_times' in dir():
    plt.figure(figsize=(10, 5))
    plt.hist(inference_times, bins=30, color='#3498db', edgecolor='white')
    plt.axvline(x=inference_times.mean(), color='red', linestyle='--', 
                label=f'Mean: {inference_times.mean():.1f}ms')
    plt.axvline(x=150, color='green', linestyle='--', label='Target: 150ms')
    plt.xlabel('Inference Time (ms)')
    plt.ylabel('Count')
    plt.title('Inference Time Distribution')
    plt.legend()
    plt.savefig(WORK_DIR / 'inference_benchmark.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7️⃣ Export Model

In [ ]:
# Export to different formats
print("📦 Exporting model...\n")

# Export to ONNX (for deployment)
print("Exporting to ONNX...")
onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True)
print(f"✅ ONNX exported: {onnx_path}")

# Export to TorchScript
print("\nExporting to TorchScript...")
torchscript_path = best_model.export(format='torchscript', imgsz=640)
print(f"✅ TorchScript exported: {torchscript_path}")

In [ ]:
# Copy best weights to easy-to-find location
FINAL_MODEL_DIR = WORK_DIR / 'final_model'
FINAL_MODEL_DIR.mkdir(exist_ok=True)

# Copy files
if BEST_WEIGHTS.exists():
    shutil.copy(BEST_WEIGHTS, FINAL_MODEL_DIR / 'best.pt')
    print(f"✅ Copied best.pt to {FINAL_MODEL_DIR}")

# Copy ONNX if exists
onnx_file = RUNS_DIR / 'weights' / 'best.onnx'
if onnx_file.exists():
    shutil.copy(onnx_file, FINAL_MODEL_DIR / 'best.onnx')
    print(f"✅ Copied best.onnx to {FINAL_MODEL_DIR}")

print(f"\n📁 Final model directory: {FINAL_MODEL_DIR}")
for f in FINAL_MODEL_DIR.iterdir():
    print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.2f} MB)")

## 8️⃣ Summary Report

In [ ]:
# Generate final report
print("="*60)
print("📊 TRAINING SUMMARY REPORT")
print("="*60)

print(f"""
🏋️ MODEL INFORMATION
--------------------
Base Model: {TRAINING_CONFIG['model']}
Image Size: {TRAINING_CONFIG['imgsz']}
Epochs: {TRAINING_CONFIG['epochs']}
Batch Size: {TRAINING_CONFIG['batch']}

📈 TRAINING METRICS
-------------------
(See final values above)

⏱️ INFERENCE SPEED
------------------
Mean Latency: {inference_times.mean():.1f} ms (Target: <150ms)
FPS: {1000/inference_times.mean():.1f}

📁 OUTPUT FILES
---------------
Best Weights: {FINAL_MODEL_DIR}/best.pt
ONNX Model: {FINAL_MODEL_DIR}/best.onnx
Training Curves: {WORK_DIR}/training_curves.png
Predictions: {WORK_DIR}/prediction_samples.png

📝 NEXT STEPS
-------------
1. Download model weights (best.pt)
2. Upload to Google Drive for team access
3. Implement inference.py (Phase 3)
4. Build heuristics.py for form feedback
""")

print("="*60)
print("✅ Training notebook complete!")
print("="*60)

## 📥 Download Models

Sau khi training xong, download các file sau:

1. **best.pt** - Main model weights
2. **best.onnx** - ONNX format for deployment
3. **results.csv** - Training metrics
4. **training_curves.png** - Loss/metrics plots

Upload lên Google Drive: `Drive:/models/yolov8_pose/`

In [ ]:
# Zip all outputs for easy download
import zipfile

OUTPUT_ZIP = WORK_DIR / 'training_outputs.zip'

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Add model files
    for f in FINAL_MODEL_DIR.iterdir():
        zf.write(f, f'final_model/{f.name}')
    
    # Add results
    if results_csv.exists():
        zf.write(results_csv, 'results.csv')
    
    # Add plots
    for plot in WORK_DIR.glob('*.png'):
        zf.write(plot, f'plots/{plot.name}')

print(f"✅ All outputs zipped: {OUTPUT_ZIP}")
print(f"   Size: {OUTPUT_ZIP.stat().st_size / 1024 / 1024:.2f} MB")
print("\n📥 Download this file and upload to Google Drive!")